# Running PreRunner

This notebook runs `PreRunner` once to generate the three genulens histogram
files used by all other example notebooks:

| File | Contents |
|------|----------|
| `mass.dat` | Lens-mass histogram |
| `rho.dat`  | Lens/source distance density table |
| `murel.dat`| Relative proper-motion histogram |

Outputs are written to `example/pre_runner_outputs/demo/`.  Run this notebook
first; all other examples load from that directory via
`HistogramDensity.from_paths`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "src" / "gapmoe").exists():
    repo_root = cwd
elif (cwd.parent / "src" / "gapmoe").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Run this notebook from the repository root or example directory.")

sys.path.insert(0, str(repo_root / "src"))

from gapmoe import PreRunner

repo_root

In [ ]:
genulens_root = (repo_root / ".." / "genulens").resolve()
if not (genulens_root / "pre_gapmoe").exists():
    raise FileNotFoundError(
        f"genulens pre_gapmoe not found at {genulens_root / 'pre_gapmoe'}. "
        "Set genulens_root to your local genulens checkout."
    )

runner = PreRunner(
    genulens_root=genulens_root,
    output_dir=repo_root / "example" / "pre_runner_outputs",
)

pre_run = runner.run(
    ra_deg=270.0,
    dec_deg=-30.0,
    run_name="demo",
    seed=123,
)

print("Output directory:", pre_run.output_dir)
for path in [pre_run.mass_path, pre_run.rho_path, pre_run.murel_path]:
    print(f"  {path.name}: {path.stat().st_size:,} bytes")

## Quick sanity check

Load `HistogramDensity` from the generated files and evaluate the prior at one
point to confirm everything is readable.

In [ ]:
from gapmoe import GalacticPrior, HistogramDensity

density = HistogramDensity.from_paths(
    pre_run.mass_path,
    pre_run.rho_path,
    pre_run.murel_path,
)
prior = GalacticPrior(density)

# ML=0.3 Msun, DL=5 kpc, DS=8 kpc, mu_N=5 mas/yr, mu_E=2 mas/yr
lp = prior.log_prob(0.3, 5.0, 8.0, 5.0, 2.0)
print(f"log prior = {lp:.4f}")